# Writing CUDA kernels

This is the part of the environment where you get **genuinely correct GPU behaviour**, not an approximation.

Numba includes a CUDA simulator that runs `@cuda.jit` kernels in the Python interpreter, with each GPU thread as a real thread. The programming model is faithful: `threadIdx`, `blockIdx`, grid-stride loops, shared memory, `syncthreads` and race conditions all behave as they would on an L4. If your kernel has a synchronisation bug, you will get the wrong answer here for exactly the reason you would get it on hardware.

What you do not get is speed. The simulator is *slow* — thousands of times slower than plain NumPy. Keep the arrays small. This is for learning how kernels work, not for measuring them.

The environment already sets `NUMBA_ENABLE_CUDASIM=1`, so everything below just runs.

In [ ]:
import numpy as np
from numba import cuda

print(f"CUDA available to numba: {cuda.is_available()}")

## 1. The execution model

A kernel is a function that runs once per **thread**. Threads are grouped into **blocks**, and blocks into a **grid**. You launch a kernel with `kernel[blocks_per_grid, threads_per_block](args)`.

Each thread works out which piece of data is *its* piece from its own coordinates. Print them and see:

In [ ]:
@cuda.jit
def show_coordinates(out):
    # Position of this thread within its block, and of the block within the grid
    tid = cuda.threadIdx.x
    bid = cuda.blockIdx.x
    bdim = cuda.blockDim.x

    # The global index this thread is responsible for
    i = bid * bdim + tid
    if i < out.size:
        out[i] = i


n = 12
out = np.zeros(n, dtype=np.int32)

# 3 blocks of 4 threads = 12 threads, one per element
show_coordinates[3, 4](out)
print(out)

`cuda.grid(1)` is shorthand for that `bid * bdim + tid` calculation, and is what you will normally write.

## 2. Vector addition

The canonical first kernel. Note the guard — `if i < out.size` — which is not optional.

In [ ]:
@cuda.jit
def vector_add(a, b, out):
    i = cuda.grid(1)
    if i < out.size:          # threads beyond the data must do nothing
        out[i] = a[i] + b[i]


n = 256
a = np.arange(n, dtype=np.float32)
b = np.arange(n, dtype=np.float32) * 10

# Move data to the device, run, bring the result back
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_out = cuda.device_array_like(a)

threads_per_block = 32
# Round *up*, so the last partial block still gets covered
blocks = (n + threads_per_block - 1) // threads_per_block
print(f"launching {blocks} blocks x {threads_per_block} threads = {blocks * threads_per_block} threads for {n} elements")

vector_add[blocks, threads_per_block](d_a, d_b, d_out)
result = d_out.copy_to_host()

print(f"correct: {np.allclose(result, a + b)}")
print(result[:8])

Two things to take away:

1. **The grid is usually bigger than the data.** 256 elements with 32 threads per block needs 8 blocks exactly, but 250 elements would need 8 blocks too — and 6 threads would run past the end of the array. The `if i < out.size` guard is what stops that being a memory error.
2. **Data has to be moved.** `cuda.to_device` and `.copy_to_host()` are real transfers on real hardware, and often dominate the runtime of small kernels. This is why "just move it to the GPU" is not automatically faster.

## 3. Race conditions are real here

Threads run concurrently, so two threads writing to the same location is a bug. Here is one, deliberately:

In [ ]:
@cuda.jit
def broken_sum(values, out):
    i = cuda.grid(1)
    if i < values.size:
        out[0] += values[i]   # BUG: every thread reads, adds, writes the same slot


values = np.ones(128, dtype=np.float32)
total = np.zeros(1, dtype=np.float32)
broken_sum[4, 32](values, total)

print(f"got {total[0]}, expected {values.sum()}")
print("(run this cell a few times - the answer may vary)")

The fix is an **atomic** operation, which makes read-modify-write indivisible:

In [ ]:
@cuda.jit
def atomic_sum(values, out):
    i = cuda.grid(1)
    if i < values.size:
        cuda.atomic.add(out, 0, values[i])


total = np.zeros(1, dtype=np.float32)
atomic_sum[4, 32](values, total)
print(f"got {total[0]}, expected {values.sum()}")

Atomics are correct but serialise contending threads. For a full reduction, shared memory is the better tool.

## 4. Shared memory and `syncthreads`

Shared memory is a small, fast scratchpad visible to every thread **in the same block**. Using it correctly requires `cuda.syncthreads()`, a barrier all threads in the block must reach before any proceeds.

This kernel sums each row of a matrix with a tree reduction:

In [ ]:
THREADS = 32


@cuda.jit
def row_sums(matrix, out):
    row = cuda.blockIdx.x          # one block per row
    tid = cuda.threadIdx.x

    partial = cuda.shared.array(shape=THREADS, dtype=np.float32)

    # Each thread sums a strided slice of the row
    acc = np.float32(0.0)
    for col in range(tid, matrix.shape[1], cuda.blockDim.x):
        acc += matrix[row, col]
    partial[tid] = acc

    cuda.syncthreads()             # (A) everyone has written their partial

    # Tree reduction: halve the active threads each round
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            partial[tid] += partial[tid + stride]
        cuda.syncthreads()         # (B) this round's writes are visible
        stride //= 2

    if tid == 0:
        out[row] = partial[0]


rows, cols = 8, 100
matrix = np.random.default_rng(0).random((rows, cols), dtype=np.float32)
sums = np.zeros(rows, dtype=np.float32)

row_sums[rows, THREADS](matrix, sums)

print(f"correct: {np.allclose(sums, matrix.sum(axis=1), rtol=1e-4)}")
print(f"kernel : {sums[:4]}")
print(f"numpy  : {matrix.sum(axis=1)[:4]}")

**Now break it.** Comment out barrier `(B)` and re-run. The answers go wrong, because threads start reading `partial[tid + stride]` before the owning thread has written it.

This is worth doing properly, because it is the single most common class of GPU bug, and because it demonstrates that the simulator is not hand-waving: it reproduces the failure for the right reason.

## 5. Two-dimensional grids

For image and matrix work, grids can be 2D. `cuda.grid(2)` returns a `(row, col)` pair.

In [ ]:
@cuda.jit
def matrix_scale(matrix, factor, out):
    row, col = cuda.grid(2)
    if row < out.shape[0] and col < out.shape[1]:
        out[row, col] = matrix[row, col] * factor


m = np.arange(24, dtype=np.float32).reshape(4, 6)
scaled = np.zeros_like(m)

threads_2d = (4, 4)
blocks_2d = (
    (m.shape[0] + threads_2d[0] - 1) // threads_2d[0],
    (m.shape[1] + threads_2d[1] - 1) // threads_2d[1],
)
matrix_scale[blocks_2d, threads_2d](m, np.float32(10), scaled)
print(scaled)

## Exercises

1. Write a kernel computing the elementwise **maximum** of two arrays.
2. Rewrite `vector_add` as a **grid-stride loop**, so it handles any array size with a fixed grid:
   ```python
   start = cuda.grid(1)
   stride = cuda.gridsize(1)
   for i in range(start, out.size, stride):
       ...
   ```
3. Remove barrier `(A)` from `row_sums` (keeping `(B)`). Does it still give the right answer? Why might it seem to, and why is it still a bug?
4. Write a kernel that reverses an array. What goes wrong if you do it in place, and how would you fix it?

## A closing reminder

The kernels above are correct CUDA and would compile and run on a real L4 unchanged. But every timing you could take in this environment is a CPU timing of an interpreter simulating threads. **Nothing here tells you which kernel is faster.** Performance work needs hardware and a profiler.